In [1]:
import pandas as pd
import boto3
from IPython.core.display_functions import display

conn = boto3.client('athena')

if not start_date:
    start_date = end_date
# TODO: these columns should be PascalCase and not snake_case
query = f"""
            SELECT
                sec_id,
                universe_type,
                model_name,
                feature,
                explainer,
                explainer_value,
                expected_value,
                last_update,
                end_date
            FROM {config.aws['athena']['databases']['mqr']}.model_explanation_data
            WHERE end_date BETWEEN DATE('{start_date.strftime('%Y-%m-%d')}') AND DATE('{end_date.strftime('%Y-%m-%d')}')
        """
if explainer:
    query += f" AND explainer = '{explainer}'"
if pillar_name:
    query += " AND model_name IN (" + f"'{pillar_name.capitalize()}Positive', " + f"'{pillar_name.capitalize()}Negative')"
if top:
    query += f""" LIMIT {top}"""
df = aws.query_athena(con, query, as_date=['end_date', 'last_update'])

shapley_df['Model'] = shapley_df['model_name'].str.slice(start=0, stop=-8)
shapley_df["explainer_value"] = shapley_df["explainer_value"].astype('float64')
shapley_df['PositiveOrNegative'] = shapley_df['model_name'].str.slice(start=-8)
shapley_df = shapley_df[['sec_id', 'universe_type', 'feature', 'explainer_value', 'end_date', 'Model', 'PositiveOrNegative']]
shapley_df = pd.pivot_table(shapley_df, index=['sec_id', 'universe_type', 'feature',  'end_date', 'Model'], columns=['PositiveOrNegative'], values='explainer_value').reset_index()
shapley_df['ShapleyValue'] = shapley_df['Positive'].fillna(0) - shapley_df['Negative'].fillna(0)
shapley_df = pd.pivot_table(shapley_df, index=['sec_id', 'universe_type', 'feature', 'end_date'], columns=['Model'], values='ShapleyValue').reset_index()
shapley_df = shapley_df.rename(
                  columns={"sec_id": "SecId", "universe_type": "UniverseType", "feature": "Feature", "end_date": "EndDate",
                  "Parent": "Parent", "People": "People", "Process": "Process"}, errors="raise")
shapley_df['EndDate'] = shapley_df['EndDate'].astype('datetime64[ns]')

if len(old_pillars_df) > 1:
    shared_sec_ids = set(df['SecId']).intersection(set(old_pillars_df['SecId']))
    combined_pillars_df = pd.concat(
        [old_pillars_df.loc[old_pillars_df['SecId'].isin(shared_sec_ids)],
         df.loc[:, ['SecId', 'EndDate', 'ParentRawModelOutput', 'PeopleRawModelOutput', 'ProcessRawModelOutput']]
         ], axis=0, sort=True, join='inner')
    trailing_average_df = combined_pillars_df.groupby('SecId', as_index=False).mean().rename(
        columns={'ParentRawModelOutput': 'ParentRawSmoothed',
                 'PeopleRawModelOutput': 'PeopleRawSmoothed',
                 'ProcessRawModelOutput': 'ProcessRawSmoothed'})
    df = df.merge(trailing_average_df, on='SecId', how='left', validate='1:1')
else:
    ValueError('Insufficient data within old_pillars_df to perform smoothing.')
display(df)



bool = True
#if config.model_parameters['postprocessing']['apply_pillar_smoothing']:
if bool:
    df = apply_pillar_smoothing(df, old_pillars_df, process_shapley_values)
    # test_df_row_count(df, row_count, 'after smoothing logic is applied')
else:
    df['ParentRawSmoothed'] = df['ParentRawModelOutput']
    df['PeopleRawSmoothed'] = df['PeopleRawModelOutput']
    df['ProcessRawSmoothed'] = df['ProcessRawModelOutput']
display(df)


shapley_df = perform_raw_pillar_score_smoothing(shapley_df, old_shapley_df, universe_config.validation['row_count'],
                                                process_shapley_values=True)

KeyboardInterrupt: 